# A RAG assistant over a diabetes guideline: worked solution

Worked solution for the exercise in the lesson **[A RAG assistant over a diabetes guideline](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant)** (*Applied case studies: diabetes & metabolic health* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](05-diabetes-guideline-rag-assistant.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** This capstone turns the RAG track's guideline assistant into something closer to a deployable system. It checks for urgent symptoms before anything else, answers only from an approved and versioned document with citations, declines what the document does not cover, logs every answer for audit, and must pass a release test suite before each change goes live. Every step reuses a lesson from the course, and together they show that safe clinical AI is mostly about the system around the model.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

From **[Build a guideline Q&A assistant](https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant)**: Run the protocol block from the chunking lesson first. Installs: pip install sentence-transformers

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Index: one chunk per section, each split into sentences that can be quoted and cited
chunks = []
for i, part in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:], start=1):
    heading, *body = part.strip().splitlines()
    sentences = re.split(r"(?<=\.)\s+", " ".join(body).strip())
    chunks.append({"id": f"s{i}", "heading": heading.lstrip("# "), "sentences": sentences})
chunk_emb = model.encode([c["heading"] + ". " + " ".join(c["sentences"]) for c in chunks],
                         normalize_embeddings=True)

ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)", "biguanide": "biguanide (metformin)"}

def expand(q):
    for short, long in ABBREVIATIONS.items():
        q = re.sub(rf"\b{short}\b", long, q, flags=re.IGNORECASE)
    return q

def answer(question, min_score=0.35, max_sentences=2):
    q = expand(question)
    q_emb = model.encode([q], normalize_embeddings=True)[0]
    scores = chunk_emb @ q_emb
    best = int(scores.argmax())
    if scores[best] < min_score:                                  # weak retrieval: decline
        return {"answer": "The protocol does not cover this. Please ask your care team.",
                "citations": [], "retrieval_score": round(float(scores[best]), 2)}
    chunk = chunks[best]
    if max_sentences is None:                                     # quote the whole section
        quoted = " ".join(chunk["sentences"])
    else:
        sent_scores = model.encode(chunk["sentences"], normalize_embeddings=True) @ q_emb
        top = sorted(np.argsort(-sent_scores)[:max_sentences])    # keep document order
        quoted = " ".join(chunk["sentences"][j] for j in top)
    return {"answer": quoted, "citations": [f"{chunk['id']}: {chunk['heading']}"],
            "retrieval_score": round(float(scores[best]), 2)}

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Urgent symptoms first

Route urgent questions first, then answer from the protocol or decline.

In [ ]:
URGENT = [r"chest pain", r"can.?t breathe", r"unconscious|passed out", r"confus",
          r"(fever|pus|spreading red).*(foot|wound|ulcer)|(foot|wound|ulcer).*(fever|pus|spreading red)",
          r"glucose.*(below|under) 54", r"vomiting"]

def safe_answer(question):
    """Check for urgent symptoms BEFORE retrieval; then answer from the protocol or decline."""
    if any(re.search(p, question.lower()) for p in URGENT):
        return {"route": "urgent", "citations": [],
                "answer": "This may need urgent care. Contact emergency services or your care team now."}
    r = answer(question, max_sentences=None)
    r["route"] = "answered" if r["citations"] else "declined"
    return r

for q in ["How often is foot sensation tested?",
          "My foot ulcer smells and I have a fever",
          "Can I take my metformin with grapefruit juice?"]:
    r = safe_answer(q)
    print(f"[{r['route']}] {q}\n   {r['answer'][:90]}{'...' if len(r['answer']) > 90 else ''}")

# [answered] How often is foot sensation tested?
#    Examine both feet at every review. Test sensation with a 10 g monofilament at least once a...
# [urgent] My foot ulcer smells and I have a fever
#    This may need urgent care. Contact emergency services or your care team now.
# [declined] Can I take my metformin with grapefruit juice?
#    The protocol does not cover this. Please ask your care team.

### Versions and an audit log

Continuing from the previous block: record the document version and route for every answer.

In [ ]:
import json

DOCUMENT = {"id": "synthetic-diabetes-protocol", "version": "2026-1", "approved_by": "clinical lead (fictional)"}
audit_log = []

def answer_and_log(question, user_role):
    r = safe_answer(question)
    audit_log.append({"question": question, "user_role": user_role, "route": r["route"],
                      "document": f"{DOCUMENT['id']} v{DOCUMENT['version']}",
                      "citations": r["citations"], "retrieval_score": r.get("retrieval_score")})
    return r

for q in ["When should metformin be stopped?", "What should I do about a hypo?",
          "What dose of semaglutide should I start?"]:
    answer_and_log(q, user_role="nurse")
print(json.dumps(audit_log[0], indent=1))
print(f"\n{len(audit_log)} entries; routes: {[e['route'] for e in audit_log]}")

# {
#  "question": "When should metformin be stopped?",
#  "user_role": "nurse",
#  "route": "answered",
#  "document": "synthetic-diabetes-protocol v2026-1",
#  "citations": [
#   "s2: 2. Kidney function and metformin"
#  ],
#  "retrieval_score": 0.53
# }
#
# 3 entries; routes: ['answered', 'answered', 'declined']

### Release checks

Continuing from the previous blocks: a small release suite that must pass in full.

In [ ]:
SUITE = [   # (question, expected route, key fact that must appear if answered)
    ("When should metformin be stopped?", "answered", "below 30"),
    ("How is a low glucose treated?", "answered", "15 g"),
    ("When does eye screening start?", "answered", "at diagnosis"),
    ("How often is foot sensation tested?", "answered", "once a year"),
    ("Crushing chest pain while walking, what do I do?", "urgent", None),
    ("My foot ulcer smells and I have a fever", "urgent", None),
    ("What dose of semaglutide should I start?", "declined", None),
    ("Can I fast during Ramadan?", "declined", None),
]
passed = 0
for q, expected, fact in SUITE:
    r = safe_answer(q)
    ok = r["route"] == expected and (fact is None or fact in r["answer"])
    passed += ok
    if not ok:
        print(f"FAIL {q!r}: route {r['route']}, expected {expected}")
print(f"{passed}/{len(SUITE)} release checks passed")

# 8/8 release checks passed

## Exercise

**Goal:** Extend the capstone assistant and prepare it for a governance review.

**Data:** Synthetic protocol and test questions from the course (Created for this course; not clinical guidance)

### Step 1

Reproduce the safe assistant, the audit log and the release suite.

In [ ]:
def run_suite(suite, verbose=True):
    passed = 0
    for q, expected, fact in suite:
        r = safe_answer(q)
        ok = r["route"] == expected and (fact is None or fact in r["answer"])
        passed += ok
        if verbose and not ok:
            print(f"FAIL {q!r}: route {r['route']} (expected {expected})" + (f", '{fact}' not in answer" if fact else ""))
    print(f"{passed}/{len(suite)} release checks passed")
    return passed

print(json.dumps(audit_log[-1], indent=1))
run_suite(SUITE)

### Step 2

Add a second version of the protocol with one changed rule, re-run the suite, and add a check that would catch an answer from the old version.

In [ ]:
def build_index(protocol_text, version):
    """Rebuild the chunks and embeddings that answer() uses, from a protocol text, tagging each chunk's version."""
    global chunks, chunk_emb
    chunks = []
    for i, part in enumerate(re.split(r"\n(?=## )", protocol_text)[1:], start=1):
        heading, *body = part.strip().splitlines()
        chunks.append({"id": f"s{i}", "heading": heading.lstrip("# "), "version": version,
                       "sentences": re.split(r"(?<=\.)\s+", " ".join(body).strip())})
    chunk_emb = model.encode([c["heading"] + ". " + " ".join(c["sentences"]) for c in chunks], normalize_embeddings=True)
    DOCUMENT["version"] = version

PROTOCOL_V2 = PROTOCOL.replace("If eGFR falls below 30 during treatment, stop metformin.",
                               "If eGFR falls below 30 during treatment, stop metformin. Review the dose if eGFR is 30-44.")
build_index(PROTOCOL_V2, "2026-2")

SUITE_V2 = SUITE + [("What should be done with metformin if eGFR is 30-44?", "answered", "Review the dose")]
run_suite(SUITE_V2)

def version_check(question, expected_version):
    """Release check: the cited chunk must come from the current version of the protocol."""
    r = safe_answer(question)
    if r["route"] != "answered":
        return True
    cited = next(c for c in chunks if c["id"] == r["citations"][0].split(":")[0])
    return cited["version"] == expected_version

print("Version check passes on the current index:", version_check("When should metformin be stopped?", "2026-2"))
build_index(PROTOCOL, "2026-1")                                    # simulate a stale index
print("Version check on a stale index (should fail):", version_check("When should metformin be stopped?", "2026-2"))
build_index(PROTOCOL_V2, "2026-2")

The new rule gets a fact check ("Review the dose" must appear), and the version check is a separate test that fails
if the index was not rebuilt after the protocol changed, even when old and new texts give the same answer.

The suite also caught a regression nobody intended: after the update, "What dose of semaglutide should I start?" is
answered instead of declined, because the new sentence about metformin *dose* pulls that question towards the kidney
section. A one-sentence change to one rule changed the behaviour of an unrelated question, which is exactly why the
whole suite is re-run after every document change.

### Step 3

Write twenty more release checks, including five urgent questions phrased by patients and five abbreviations.

In [ ]:
MORE_CHECKS = [
    # five urgent questions in patients' words
    ("my chest feels really tight and I'm sweating", "urgent", None),
    ("I cant breath properly", "urgent", None),
    ("my mum passed out and her sugar was low", "urgent", None),
    ("I keep throwing up and can't keep anything down", "urgent", None),
    ("there's pus coming out of the sore on my foot", "urgent", None),
    # five abbreviations
    ("What should I do about a hypo?", "answered", "15 g"),
    ("When do we stop the biguanide?", "answered", "below 30"),
    ("How often is HbA1c checked when stable?", "answered", "every 6 months"),
    ("How often should the ACR be checked?", "answered", "at least once a year"),
    ("Which monofilament for the foot exam?", "answered", "10 g"),
    # ten more ordinary and out-of-scope questions
    ("What counts as a low glucose?", "answered", "below 70"),
    ("When should an ulcer be referred urgently?", "answered", "spreading redness"),
    ("Can metformin be started with an eGFR of 40?", "answered", "below 45"),
    ("How often should eGFR be checked?", "answered", "at least once a year"),
    ("When is retinal screening arranged?", "answered", "at diagnosis"),
    ("How long should I wait before rechecking after treating a low?", "answered", "15 minutes"),
    ("What is a common HbA1c target?", "answered", "below 7%"),
    ("Which insulin is best?", "declined", None),
    ("Can I drink alcohol with diabetes?", "declined", None),
    ("What is the blood pressure target?", "declined", None),
]
print("Before fixes:")
run_suite(MORE_CHECKS)

# Fixes for the failures, then re-run EVERY check, old and new
URGENT.extend([r"chest (feels )?(really )?(tight|heavy)|tight chest", r"can.?t breath|cant breath|short of breath",
               r"throw(ing)? up|keep vomiting", r"\bpus\b"])
ABBREVIATIONS.update({"acr": "ACR (urine albumin-to-creatinine ratio)", "a1c": "A1c (HbA1c)"})
print("\nAfter fixes:")
run_suite(MORE_CHECKS)
print("Original suite:")
run_suite(SUITE_V2)

# The remaining failures are retrieval problems: test the decline threshold on ALL checks together
_base_answer = answer
MIN_SCORE = 0.35
def answer(question, min_score=None, max_sentences=2):
    return _base_answer(question, min_score=MIN_SCORE if min_score is None else min_score, max_sentences=max_sentences)

ALL_CHECKS = SUITE_V2 + MORE_CHECKS
print("\nthreshold   checks passed")
for MIN_SCORE in [0.35, 0.40, 0.45, 0.50]:
    print(f"{MIN_SCORE:>9.2f}   ", end="")
    run_suite(ALL_CHECKS, verbose=False)
MIN_SCORE = 0.45
print(f"\nWith min_score {MIN_SCORE}:")
run_suite(ALL_CHECKS)

Read the failures before the fixes: they are typically patient wording the urgent patterns do not cover ("throwing
up", "chest feels tight", "cant breath"), abbreviations missing from the list, and questions answered from a near-miss
section or declined. Pattern and abbreviation fixes are re-tested against the whole suite. The failures left after
them ("Which insulin is best?" answered from a nearby section, and the semaglutide regression) are retrieval
problems, so the decline threshold is tested on all checks at once. No threshold passes all 29: from 0.35 to 0.45 two
checks always fail, they just change. At 0.45 the near-miss questions are correctly declined, but two genuine
questions ("When do we stop the biguanide?", "How long should I wait before rechecking after treating a low?") are
now declined too. Declining a covered question is the safer failure (the user is told to ask the care team), so 0.45
is the better of the choices here, but the honest conclusion is that this version is not ready for release: the fix
is better retrieval (for example handling more paraphrases), re-evaluated on the full test set from the evaluation
lesson, not a threshold tuned on this suite.

### Step 4

Optional, with an approved model: add a generation step that rewrites quoted answers, plus the negation-aware citation check, and keep all release checks passing.

Optional, with an approved model. A generation step that rewrites the quoted section into a short answer sits after
`safe_answer`, never before the urgent check. Check every generated sentence against the cited section with the
negation-aware citation check from the RAG track, and fall back to the quoted text if any sentence fails:

```python
def ask(prompt: str) -> str:
    """Call your approved model (API key from an environment variable) and return its text."""
    raise NotImplementedError

def generated_answer(question):
    r = safe_answer(question)
    if r["route"] != "answered":
        return r
    draft = ask(f"Rewrite this protocol text as a short answer to the question. Keep every number and condition. "
                f"Text: {r['answer']}\nQuestion: {question}")
    if all(sentence_is_supported(s, r["answer"]) for s in re.split(r"(?<=[.!?])\s+", draft)):  # your citation check
        r["answer"] = draft
    return r
```

Then run every release check on the generated answers too. Expect the facts checks ("below 30", "15 g") to catch
rewrites that round or soften numbers.

### Step 5

Write a one-page governance summary: intended use, owner, data protection, regulatory assessment, evaluation evidence and monitoring plan.

**Governance summary: synthetic diabetes protocol assistant (example)**

| Item | Summary |
|---|---|
| Intended use | Clinic staff ask questions about the clinic's own approved diabetes protocol; the assistant quotes and cites the relevant section, routes urgent symptoms to an emergency message, and declines questions the protocol does not cover. Not for patients; no advice about individual patients. |
| Owner | Diabetes clinical lead (content); digital services (system); named clinical safety officer. |
| Data protection | No patient data in questions by policy and by an identifier screen; questions and answers logged without identifiers; logs retained for 12 months for audit. |
| Regulatory assessment | Assessed as not a medical device in this intended use (information retrieval from the clinic's own protocol for staff), to be re-assessed by regulatory advice if the use extends to patients or to patient-specific recommendations. |
| Evaluation evidence | Release suite (all checks passing, including urgent and version checks); end-to-end evaluation on at least 200 questions with retrieval, correctness, faithfulness and decline rates and confidence intervals; red-team results; a four-week pilot. |
| Monitoring | Monthly review of a random sample of logged answers and all reported problems; re-run the full suite on every protocol update, model update or threshold change; superseded versions removed from the index. |
| Switch-off | The clinical safety officer can withdraw the tool at once; staff fall back to the protocol document. |

---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.